# 🧠 DriftMind: Cold-Start & Online Forecasting Demo

Welcome to the official demonstration of the **DriftMind Python Client (v0.5.0)**.

This notebook walks through a complete online forecasting pipeline: creating a forecaster, feeding synthetic time-series data with drifts, collecting one-step-ahead predictions, and analyzing accuracy, anomaly scores, and cluster evolution.

### 📋 Table of Contents
1. [**Credentials & Client Setup**](#1.-Credentials-&-Client-Setup)
2. [**Create a New Forecaster**](#2.-Create-a-New-Forecaster)
3. [**Synthetic Data Generation**](#3.-Synthetic-Data-Generation)
4. [**Phase 1: Bulk Warm-up**](#4.-Phase-1:-Bulk-Warm-up)
5. [**Phase 2: Online Forecasting**](#5.-Phase-2:-Online-Forecasting)
6. [**Accuracy Analysis**](#6.-Accuracy-Analysis)
7. [**Interactive Exploration**](#7.-Interactive-Exploration)
8. [**Drift & Anomaly Monitoring**](#8.-Drift-&-Anomaly-Monitoring)
9. [**Inspect Forecaster Details**](#9.-Inspect-Forecaster-Details)
10. [**Cleanup**](#10.-Cleanup)

In [ ]:
%pip install driftmind[examples] --quiet

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

from driftmind import DriftMindClient
from driftmind.exceptions import DriftMindConfigError, DriftMindError
from driftmind.utils import (
    generate_sin_cos_tan_with_drifts,
    load_credentials,
    plot_actual_vs_predicted,
    plot_time_series,
)

## 1. Credentials & Client Setup

Credentials are expected in the environment as:

- `DRIFTMIND_API_KEY`
- `DRIFTMIND_API_URL`

Optionally, they can be loaded from a `.env` file for local runs.

In [ ]:
try:
    creds = load_credentials()
    api_key = creds["DRIFTMIND_API_KEY"]
    base_url = creds["DRIFTMIND_API_URL"]
    display(Markdown("**Credentials loaded.**"))
except DriftMindConfigError as err:
    raise RuntimeError(
        "Failed to load DriftMind credentials. "
        "Ensure DRIFTMIND_API_KEY and DRIFTMIND_API_URL are set in "
        "your environment or .env file."
    ) from err

# Create client (no context manager in notebooks for convenience)
client = DriftMindClient(api_key=api_key, base_url=base_url)
display(Markdown(f"**Client ready** — connected to `{base_url}`"))

## 2. Create a New Forecaster

Define a minimal forecaster with three features (`sin`, `cos`, `tan`) and a small input window. DriftMind will learn patterns online as data arrives.

In [ ]:
columns = ["sin", "cos", "tan"]

forecaster_payload = {
    "forecaster_name": "Demo v0.5.0",
    "features": columns,
    "input_size": 15,
    "output_size": 1,
}

try:
    forecaster_info = client.create_forecaster(forecaster_payload)
    forecaster_id = forecaster_info["forecaster_id"]
except DriftMindError as err:
    raise RuntimeError(f"Failed to create forecaster: {err}") from err

display(Markdown(f"**Forecaster created:** `{forecaster_id}`"))
pd.DataFrame(
    [
        {"Parameter": "Name", "Value": forecaster_info["forecaster_name"]},
        {"Parameter": "Features", "Value": ", ".join(forecaster_info["features"])},
        {"Parameter": "Input size", "Value": forecaster_info["configuration"]["input_size"]},
        {"Parameter": "Output size", "Value": forecaster_info["configuration"]["output_size"]},
    ]
).style.hide(axis="index")

## 3. Synthetic Data Generation

We now generate a dataset with sinusoidal drifts using a helper from `driftmind.utils.demo`.

It creates a pandas DataFrame with columns:
- `sequence`: Integer time index from 0 to n-1.
- `sin`: Sine component with drifts.
- `cos`: Cosine component with drifts.
- `tan`: Clipped tangent component with drifts.

In [ ]:
df = generate_sin_cos_tan_with_drifts(n=600, noise_std=0.05, seed=42)
df.head()

## 4. Phase 1: Bulk Warm-up

Feed the first 100 data points in a single batch to initialize the model before switching to the online streaming loop. This gives DriftMind enough history to start producing meaningful forecasts.

In [ ]:
results = {col: [] for col in columns}
global_results = {"timestamp": [], "anomaly_score": [], "number_of_clusters": []}
running_mse = {col: 0.0 for col in columns}
count = 0

warmup_data = {col: df.iloc[:100][col].tolist() for col in columns}
client.feed_point(forecaster_id, warmup_data)
display(Markdown("**Warm-up complete** — 100 points fed in bulk."))

## 5. Phase 2: Online Forecasting

Stream the remaining 500 points one at a time. For each point, we feed it to the forecaster, request a one-step-ahead prediction, and record the squared error. Progress is logged every 100 steps.

In [ ]:
for _, row in df.iloc[100:].iterrows():
    seq = int(row["sequence"])
    point = {col: [float(row[col])] for col in columns}

    try:
        client.feed_point(forecaster_id, point)
        yhat = client.forecast(forecaster_id)
    except DriftMindError:
        continue

    if not yhat:
        continue

    count += 1
    global_results["timestamp"].append(seq)
    global_results["anomaly_score"].append(yhat.get("anomaly_score"))
    global_results["number_of_clusters"].append(yhat.get("number_of_clusters"))

    features_map = yhat.get("features", {})
    for var in columns:
        feat = features_map.get(var, {})
        preds = feat.get("predictions", [])
        if preds:
            pred_val = float(preds[0])
            exp_val = float(row[var])
            sq_err = (exp_val - pred_val) ** 2
            running_mse[var] += sq_err
            results[var].append(
                {"expected": exp_val, "predicted": pred_val, "timestamp": seq, "sq_err": sq_err}
            )

    if seq % 100 == 0:
        mse_summary = " | ".join(
            [f"**{k}**: {v / count:.4f}" for k, v in running_mse.items()]
        )
        display(Markdown(f"Sequence {seq} — {mse_summary}"))

display(Markdown(f"**Online loop complete.** {count} predictions collected."))

# Final MSE summary
mse_table = pd.DataFrame(
    [{"Feature": k, "MSE": v / count} for k, v in running_mse.items()]
)
display(Markdown("### Final MSE"))
mse_table.style.hide(axis="index").format({"MSE": "{:.6f}"})

## 6. Accuracy Analysis

Plot actual vs. predicted values for each feature, followed by the expanding-mean MSE to visualize how prediction error converges as the model learns online.

In [ ]:
for var in columns:
    df_var = pd.DataFrame(results[var])
    if df_var.empty:
        print(f"No prediction data collected for {var}.")
        continue
    plot_actual_vs_predicted(df_var, variable_name=var)

In [ ]:
plt.figure(figsize=(10, 4))
for var in columns:
    df_res = pd.DataFrame(results[var])
    plt.plot(df_res["timestamp"], df_res["sq_err"].expanding().mean(), label=f"{var} MSE")
plt.title("MSE Convergence (Online Learning)")
plt.legend()
plt.show()

## 7. Interactive Exploration

Interactive Plotly charts for zooming and hovering over the actual vs. predicted traces. Each feature is shown in a separate subplot with a shared time axis.

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True)
for i, var in enumerate(columns):
    df_v = pd.DataFrame(results[var])
    fig.add_trace(
        go.Scatter(x=df_v["timestamp"], y=df_v["expected"], name=f"{var} Actual"), row=i + 1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_v["timestamp"], y=df_v["predicted"], name=f"{var} Forecast"),
        row=i + 1,
        col=1,
    )
fig.show()

## 8. Drift & Anomaly Monitoring

Track the global anomaly score and number of active clusters over time. Spikes in the anomaly score indicate regime changes (drifts) in the data, while the cluster count reflects how the model adapts its internal representation.

In [ ]:
df_global = pd.DataFrame(global_results)

if df_global.empty:
    print("No global metrics collected.")
else:
    plot_time_series(
        df_global["timestamp"],
        df_global["anomaly_score"],
        title="Global Anomaly Score over Time",
        xlabel="Time Step",
        ylabel="Anomaly Score",
    )

    plot_time_series(
        df_global["timestamp"],
        df_global["number_of_clusters"],
        title="Global Number of Clusters over Time",
        xlabel="Time Step",
        ylabel="Number of Clusters",
    )

## 9. Inspect Forecaster Details

Query the API for the forecaster's current state: its configuration and per-feature statistics including cluster counts, observation totals, and anomaly scores.

In [ ]:
details = client.get_forecaster_details(forecaster_id)

display(Markdown(f"### Forecaster: `{details['forecaster_name']}`"))

# Configuration table
config = details["configuration"]
config_df = pd.DataFrame(
    [{"Setting": k, "Value": v} for k, v in config.items()]
)
display(Markdown("**Configuration**"))
display(config_df.style.hide(axis="index"))

# Per-feature statistics table
rows = []
for feature, stats in details["features"].items():
    rows.append({
        "Feature": feature,
        "Active Clusters": stats["active_clusters"],
        "Observations": stats["total_observations"],
        "Anomaly Score": stats["anomaly_score"],
        "Last Addition": stats["last_addition"],
    })

display(Markdown("**Feature Statistics**"))
pd.DataFrame(rows).style.hide(axis="index").format({"Anomaly Score": "{:.4f}"})

## 10. Cleanup

Delete the forecaster and close the client session to release resources.

In [ ]:
client.delete_forecaster(forecaster_id)
client.close()
display(Markdown(f"**Forecaster `{forecaster_id}` deleted.** Session closed."))